# Kimi Audio Few-Shot AD Detection

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"]     = "/root/autodl-tmp/LLM_Model"

import json
import tempfile
from pathlib import Path

import torch
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-04-01 06:48:51.015 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-04-01 06:48:51.016 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-01 06:48:51.017 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-04-01 06:48:59.280 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-01 06:48:59.282 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-01 06:49:00.220 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-01 06:49:00.403 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)


def classify_audio(wav_path: Path, control_wavs: list, dementia_wavs: list) -> str:
    """Classify a single audio file with 6-shot examples (6 Control + 6 Dementia)."""
    messages = [
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[0])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[1])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[2])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[3])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[4])},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[5])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[0])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[1])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[2])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[3])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[4])},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[5])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user",      "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text", max_new_tokens=256)
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [5]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result")


def get_examples(df, audio_dir, n=6):
    """Pick the first n Control and first n Dementia samples as few-shot examples."""
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    examples = {}
    for ad_val, label in label_map.items():
        rows = df[df["ad"] == ad_val].iloc[:n]
        wavs, ids = [], []
        for _, row in rows.iterrows():
            matches = list(audio_dir.glob(f"{label}/{row['session_id']}.*"))
            wavs.append(ensure_wav(matches[0]))
            ids.append(row["session_id"])
        examples[label] = {"session_ids": ids, "wavs": wavs}
    print(f"  Few-shot examples: Control={examples['Control']['session_ids']}, Dementia={examples['Dementia']['session_ids']}")
    return examples


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    # Select few-shot examples from this dataset
    examples = get_examples(df, audio_dir)
    example_ids = set(examples["Control"]["session_ids"] + examples["Dementia"]["session_ids"])
    control_wavs  = examples["Control"]["wavs"]
    dementia_wavs = examples["Dementia"]["wavs"]

    predictions, skipped = [], 0

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        # Skip few-shot example samples
        if row["session_id"] in example_ids:
            skipped += 1
            continue
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]), control_wavs, dementia_wavs)
            pred = parse_prediction(raw)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raw, pred = "OOM", None
        except Exception as e:
            raw, pred = str(e), None
        finally:
            torch.cuda.empty_cache()
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1) * 100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) * 100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [6]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-raw:   0%|          | 0/551 [00:00<?, ?it/s]

Pitt-raw: 100%|██████████| 551/551 [20:08<00:00,  2.19s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-raw.csv
[Pitt-raw]
  Accuracy:    63.49%
  F1:          0.6587
  Control Acc: 64.44%
  Dementia Acc:62.75%
  Valid: 545/551  Skipped: 6


In [7]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-raw: 100%|██████████| 74/74 [01:19<00:00,  1.08s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-raw.csv
[Lu-raw]
  Accuracy:    57.35%
  F1:          0.4528
  Control Acc: 81.82%
  Dementia Acc:34.29%
  Valid: 68/74  Skipped: 6


In [8]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-Demucs: 100%|██████████| 551/551 [20:06<00:00,  2.19s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-Demucs.csv
[Pitt-Demucs]
  Accuracy:    64.59%
  F1:          0.6892
  Control Acc: 57.74%
  Dementia Acc:69.93%
  Valid: 545/551  Skipped: 6


In [9]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-Demucs: 100%|██████████| 74/74 [01:30<00:00,  1.22s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-Demucs.csv
[Lu-Demucs]
  Accuracy:    70.59%
  F1:          0.6552
  Control Acc: 87.88%
  Dementia Acc:54.29%
  Valid: 68/74  Skipped: 6


In [10]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-Denoiser: 100%|██████████| 551/551 [16:52<00:00,  1.84s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-Denoiser.csv
[Pitt-Denoiser]
  Accuracy:    66.42%
  F1:          0.6955
  Control Acc: 64.02%
  Dementia Acc:68.30%
  Valid: 545/551  Skipped: 6


In [11]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-Denoiser: 100%|██████████| 74/74 [01:17<00:00,  1.04s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-Denoiser.csv
[Lu-Denoiser]
  Accuracy:    70.59%
  F1:          0.7059
  Control Acc: 72.73%
  Dementia Acc:68.57%
  Valid: 68/74  Skipped: 6


In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-FRCRN_SE: 100%|██████████| 551/551 [16:51<00:00,  1.84s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-FRCRN_SE.csv
[Pitt-FRCRN_SE]
  Accuracy:    65.14%
  F1:          0.6885
  Control Acc: 60.67%
  Dementia Acc:68.63%
  Valid: 545/551  Skipped: 6


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-FRCRN_SE: 100%|██████████| 74/74 [01:16<00:00,  1.04s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-FRCRN_SE.csv
[Lu-FRCRN_SE]
  Accuracy:    63.24%
  F1:          0.5098
  Control Acc: 90.91%
  Dementia Acc:37.14%
  Valid: 68/74  Skipped: 6


In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-MossFormer: 100%|██████████| 551/551 [16:53<00:00,  1.84s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-MossFormer.csv
[Pitt-MossFormer]
  Accuracy:    67.16%
  F1:          0.7172
  Control Acc: 58.16%
  Dementia Acc:74.18%
  Valid: 545/551  Skipped: 6


In [15]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-MossFormer: 100%|██████████| 74/74 [01:17<00:00,  1.04s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-MossFormer.csv
[Lu-MossFormer]
  Accuracy:    73.53%
  F1:          0.7188
  Control Acc: 81.82%
  Dementia Acc:65.71%
  Valid: 68/74  Skipped: 6


In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-Resemble: 100%|██████████| 551/551 [20:06<00:00,  2.19s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-Resemble.csv
[Pitt-Resemble]
  Accuracy:    65.87%
  F1:          0.6931
  Control Acc: 62.34%
  Dementia Acc:68.63%
  Valid: 545/551  Skipped: 6


In [17]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True
  Few-shot examples: Control=['F22_000', 'F22_001', 'F26_000'], Dementia=['F01_000', 'F02_000', 'F03_000']


Lu-Resemble: 100%|██████████| 74/74 [01:30<00:00,  1.22s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Lu-Resemble.csv
[Lu-Resemble]
  Accuracy:    67.65%
  F1:          0.6333
  Control Acc: 81.82%
  Dementia Acc:54.29%
  Valid: 68/74  Skipped: 6
